In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
print(PROJECT_ROOT)

c:\Users\clark\OneDrive\Desktop\drowsiness_project


In [5]:
from pathlib import Path

FRAMES_ROOT = PROJECT_ROOT / "processed" / "FRAMES"

for cls in ["alert", "low_vigilant", "drowsy"]:
    (FRAMES_ROOT / cls).mkdir(parents=True, exist_ok=True)

print("Folders created!")

Folders created!


In [ ]:
import shutil      #copying nthu alert datset to frames
from pathlib import Path
from tqdm import tqdm

nthu_root = PROJECT_ROOT / "processed" / "NTHU"

# alert
for img in tqdm((nthu_root / "alert").glob("*.jpg")):
    dst = FRAMES_ROOT / "alert" / f"nthu_{img.name}"
    shutil.copy2(img, dst)

# drowsy
for img in tqdm((nthu_root / "drowsy").glob("*.jpg")):
    dst = FRAMES_ROOT / "drowsy" / f"nthu_{img.name}"
    shutil.copy2(img, dst)

print("NTHU copied.")

29646it [01:02, 472.18it/s]
0it [00:00, ?it/s]

NTHU copied.


In [9]:
for img in tqdm((nthu_root / "drowsy").rglob("*.jpg")): #copying nthu drowsy dataset to frames
    dst = FRAMES_ROOT / "drowsy" / f"nthu_{img.name}"
    shutil.copy2(img, dst)

print("NTHU drowsy copied.")

35275it [00:54, 641.72it/s]

NTHU drowsy copied.


In [12]:
from pathlib import Path

nthu_root = PROJECT_ROOT / "processed" / "NTHU"

alert_count = len(list((nthu_root / "alert").glob("*.jpg")))
drowsy_count = len(list((nthu_root / "drowsy").rglob("*.jpg")))

print(f"NTHU Alert images  : {alert_count}")
print(f"NTHU Drowsy images : {drowsy_count}")
print(f"Total NTHU images  : {alert_count + drowsy_count}")

print("FRAMES Alert        :", len(list((FRAMES_ROOT / "alert").glob("*.jpg"))))
print("FRAMES Drowsy       :", len(list((FRAMES_ROOT / "drowsy").glob("*.jpg"))))
print("FRAMES Low Vigilant :", len(list((FRAMES_ROOT / "low_vigilant").glob("*.jpg"))))

NTHU Alert images  : 29646
NTHU Drowsy images : 35275
Total NTHU images  : 64921
FRAMES Alert        : 29646
FRAMES Drowsy       : 35275
FRAMES Low Vigilant : 0


In [13]:
import shutil
from tqdm import tqdm

mendeley_root = PROJECT_ROOT / "processed" / "MENDELEY"

# Copy alert images
for img in tqdm((mendeley_root / "alert").glob("*.jpg")):
    dst = FRAMES_ROOT / "alert" / f"mendeley_{img.name}"
    shutil.copy2(img, dst)

# Copy drowsy images
for img in tqdm((mendeley_root / "drowsy").glob("*.jpg")):
    dst = FRAMES_ROOT / "drowsy" / f"mendeley_{img.name}"
    shutil.copy2(img, dst)

print("Mendeley copied successfully!")

435it [00:00, 629.39it/s]
124it [00:00, 466.05it/s]

Mendeley copied successfully!


In [14]:
print("Alert images:",
      len(list((FRAMES_ROOT / "alert").glob("*.jpg"))))

print("Drowsy images:",
      len(list((FRAMES_ROOT / "drowsy").glob("*.jpg"))))

print("Low vigilant images:",
      len(list((FRAMES_ROOT / "low_vigilant").glob("*.jpg"))))

Alert images: 30081
Drowsy images: 35399
Low vigilant images: 0


In [15]:
for cls in ["alert", "drowsy", "low_vigilant"]:
    print(
        cls,
        len(list((FRAMES_ROOT / cls).glob("*.jpg")))
    )

alert 30081
drowsy 35399
low_vigilant 0


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = []

for cls in ["alert", "drowsy"]:
    for img in (FRAMES_ROOT / cls).glob("*.jpg"):
        data.append({
            "filepath": str(img),
            "label": cls
        })

df = pd.DataFrame(data)

print(df["label"].value_counts())

label
drowsy    35399
alert     30081
Name: count, dtype: int64


In [17]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

train_df.to_csv(PROJECT_ROOT / "train.csv", index=False)
val_df.to_csv(PROJECT_ROOT / "val.csv", index=False)
test_df.to_csv(PROJECT_ROOT / "test.csv", index=False)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 52384
Val: 6548
Test: 6548


In [18]:
print("\nTrain")
print(train_df["label"].value_counts())

print("\nValidation")
print(val_df["label"].value_counts())

print("\nTest")
print(test_df["label"].value_counts())


Train
label
drowsy    28319
alert     24065
Name: count, dtype: int64

Validation
label
drowsy    3540
alert     3008
Name: count, dtype: int64

Test
label
drowsy    3540
alert     3008
Name: count, dtype: int64


In [25]:
import pandas as pd

train_df = pd.read_csv(PROJECT_ROOT / "train.csv")
val_df = pd.read_csv(PROJECT_ROOT / "val.csv")
test_df = pd.read_csv(PROJECT_ROOT / "test.csv")

In [29]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [34]:
import importlib
import src.dataset

importlib.reload(src.dataset)

from src.dataset import DrowsinessDataset

In [35]:
import src.dataset as ds
print(dir(ds))

['Dataset', 'DrowsinessDataset', 'Image', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']


In [36]:
train_dataset = DrowsinessDataset(
    train_df,
    train_transform
)

val_dataset = DrowsinessDataset(
    val_df,
    val_transform
)

test_dataset = DrowsinessDataset(
    test_df,
    val_transform
)

In [37]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [38]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)
print(labels[:10])

c:\Users\clark\OneDrive\Desktop\drowsiness_project\venv\Lib\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


torch.Size([32, 3, 224, 224])
torch.Size([32])
tensor([1, 1, 0, 1, 1, 0, 0, 1, 1, 1])


In [39]:
import torch
import torch.nn as nn
from torchvision import models

In [42]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

!nvidia-smi

2.12.1+cpu
None
False
0
Sat Jun 27 10:02:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.08                 Driver Version: 596.08         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   57C    P0             13W /   50W |       0MiB /   8151MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------

In [40]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cpu
